In [1]:
import pandas as pd
import joblib
import json

# Auto-select best model from metadata
with open("te_model_metadata.json", "r") as f:
    metadata_te = json.load(f)

model_name_te = metadata_te["model_name"]
print(f"Using {model_name_te.upper()} model (RF RMSE: {metadata_te['rf_rmse']:.4f}, XGB RMSE: {metadata_te['xgb_rmse']:.4f})")

model_te = joblib.load(f"best_{model_name_te}_te_model.joblib")
with open(f"{model_name_te}_te_feature_cols.json", "r") as f:
    feature_cols_te = json.load(f)

# Load prediction data
pred_te = pd.read_csv("../data/processed/te_pred_combined.csv")

# Generate predictions
identifiers_te = pred_te[["player_id", "player_display_name", "college_flag"]]
X_pred_te = pred_te[feature_cols_te]

predictions_te = model_te.predict(X_pred_te)

results_te = identifiers_te.copy()
results_te["predicted_weekly_ppr"] = predictions_te.round(2)
results_te = results_te.sort_values("predicted_weekly_ppr", ascending=False).reset_index(drop=True)

print(results_te.head(20))

results_te.to_csv("../data/processed/te_predictions_2026.csv", index=False)
print("\nExport complete")
print(f"Total players predicted: {len(results_te)}")
print(f"NFL players: {(results_te['college_flag'] == 0).sum()}")
print(f"CFB players: {(results_te['college_flag'] == 1).sum()}")

Using XGB model (RF RMSE: 2.5592, XGB RMSE: 2.5395)
     player_id player_display_name  college_flag  predicted_weekly_ppr
0   00-0037744        Trey McBride             0                 14.33
1   00-0039065         Sam LaPorta             0                 11.82
2   00-0033288       George Kittle             0                 11.64
3   00-0039338        Brock Bowers             0                 11.44
4   00-0036970          Kyle Pitts             0                 11.17
5   00-0030506        Travis Kelce             0                 10.89
6   00-0040128        Tyler Warren             0                  9.87
7   00-0036040       Juwan Johnson             0                  9.84
8   00-0038996        Tucker Kraft             0                  9.49
9   00-0040663   Harold Fannin Jr.             0                  9.21
10  00-0030061           Zach Ertz             0                  8.98
11  00-0034383      Dalton Schultz             0                  8.91
12  00-0038041       Jake